# Práctica voluntariado

## 1. Leer y unir

In [1]:
import json
import unicodedata
from pathlib import Path

import pandas as pd

RAW = Path("../datos/raw")
print(sorted(p.name for p in RAW.iterdir()))

['inscricions.csv', 'quendas.csv']


In [2]:
quendas = pd.read_csv(RAW / "quendas.csv", sep=";", decimal=",")
inscricions = pd.read_csv(RAW / "inscricions.csv")

for nome, df in [("quendas", quendas), ("inscricions", inscricions)]:
    print(f"{nome:11s} {df.shape}  columnas: {list(df.columns)}")

quendas     (704, 5)  columnas: ['data', 'quenda', 'tarefa', 'email', 'horas']
inscricions (140, 5)  columnas: ['nome', 'email', 'data_nacemento', 'concello', 'data_alta']


In [3]:
quendas.head()

,data,quenda,tarefa,email,horas
0,02/10/2026,mañá,Reparto,elena.souto@correo.example,3.5
1,06/09/2026,mañá,Clasificación,roi.cotelo@correo.example,2.5
2,25/10/2026,mañá,Clasificación,iria.varela@correo.example,2.5
3,30/10/2026,mañá,Clasificación,elena.rial@correo.example,3.0
4,02/09/2026,mañá,Recollida en supermercados,xurxo.fernandez98@correo.example,4.0


In [4]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   data    704 non-null    str    
 1   quenda  704 non-null    str    
 2   tarefa  704 non-null    str    
 3   email   704 non-null    str    
 4   horas   704 non-null    float64
dtypes: float64(1), str(4)
memory usage: 27.6 KB


In [5]:
inscricions.head()

,nome,email,data_nacemento,concello,data_alta
0,Tareixa Pena,tareixa.pena@correo.example,2006-11-20,A Coruña,2026-01-10
1,Rosa Calvo,rosa.calvo@correo.example,2002-11-01,A Coruña,2026-04-12
2,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15
3,Roi Lema,roi.lema@correo.example,1982-10-08,A Coruña,2024-12-04
4,Uxío Rial,uxio.rial@correo.example,1973-07-07,Culleredo,2026-01-15


In [6]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nome            140 non-null    str  
 1   email           140 non-null    str  
 2   data_nacemento  140 non-null    str  
 3   concello        140 non-null    str  
 4   data_alta       140 non-null    str  
dtypes: str(5)
memory usage: 5.6 KB


In [7]:
# quendas["data"] = pd.to_datetime(quendas["data"], format="%d/%m/%Y")
# inscricions["data_alta"] = pd.to_datetime(inscricions["data_alta"], format="%Y-%m-%d")
# inscricions["data_nacemento"] = pd.to_datetime(inscricions["data_nacemento"], format="%Y-%m-%d")

#nomaliez hay 3 errores - hya varios que no cruzan, coreo en vez de correo , algo de puntos
#Convertir fechas con comentarios sobre el funcionamiento del código

def converter_datas(serie, formatos):
    """Proba cada formato por orde e devolve as datas e os textos que non encaixaron en ningún."""
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]") #serie vacía para guardar los resultados
    for formato in formatos: #intentamos convertir cada una de las fechas a formato datetime con cada uno de los formatos indicados
        intento = pd.to_datetime(serie, format=formato, errors="coerce") #si un valor no encaja con el formato, lo convierte en NaT
        resultado = resultado.fillna(intento) #guarda el intento que se ha podido convertir
    return resultado, serie[resultado.isna()]

quendas["data"], fallos = converter_datas(quendas["data"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_alta"], fallos = converter_datas(inscricions["data_alta"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_nacemento"], fallos = converter_datas(inscricions["data_nacemento"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])

print("Sin convertir:", fallos.tolist())#imprimimos intentos que no se han podido convertir con ninguno de los formatos para revisar

Sin convertir: []


In [8]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nome            140 non-null    str           
 1   email           140 non-null    str           
 2   data_nacemento  140 non-null    datetime64[ns]
 3   concello        140 non-null    str           
 4   data_alta       140 non-null    datetime64[ns]
dtypes: datetime64[ns](2), str(3)
memory usage: 5.6 KB


In [9]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    704 non-null    datetime64[ns]
 1   quenda  704 non-null    str           
 2   tarefa  704 non-null    str           
 3   email   704 non-null    str           
 4   horas   704 non-null    float64       
dtypes: datetime64[ns](1), float64(1), str(3)
memory usage: 27.6 KB


In [10]:
horastotales=quendas["horas"].sum()
horastotales

np.float64(2090.5)

Normaliza a clave (o correo) nas dúas táboas.

In [11]:
print(sorted(quendas["email"].unique())[:12])

[' alba.cotelo96@correo.example ', ' alba.cotelo@correo.example ', ' alba.rey@correo.example ', ' ana.souto@correo.example ', ' ana.vilar@correo.example ', ' brais.cotelo@correo.example ', ' brais.dopico@correo.example ', ' brais.rey@correo.example ', ' carme.bermudez@correo.example ', ' carme.calvo27@correo.example ', ' carme.cotelo@correo.example ', ' carme.pena@correo.example ']


In [12]:
print(sorted(inscricions["email"].unique())[:12])

['alba.calvo@correo.example', 'alba.castro60@correo.example', 'alba.castro@correo.example', 'alba.cotelo59@correo.example', 'alba.cotelo96@correo.example', 'alba.cotelo@correo.example', 'alba.mourino@correo.example', 'alba.rey@correo.example', 'ana.calvo@correo.example', 'ana.lema@correo.example', 'ana.souto@correo.example', 'ana.vilar@correo.example']


In [13]:
def normalizar(texto):
    """Minúsculas, sen tiles e sen espazos sobrantes."""
    texto = unicodedata.normalize("NFKD", str(texto).strip().lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())

inscricions["clave_email"] = inscricions["email"].map(normalizar)
quendas["clave_email"] = quendas["email"].map(normalizar)
print("Emails distintos en inscricions:", inscricions["email"].nunique(), "→ claves:", inscricions["clave_email"].nunique())
print("Emails distintos en quendas:", quendas["email"].nunique(), "→ claves:", quendas["clave_email"].nunique())


Emails distintos en inscricions: 140 → claves: 140
Emails distintos en quendas: 305 → claves: 113


Une as quendas coas inscricións de forma que non se perda ningunha quenda e que cada quenda case cunha soa persoa.

In [14]:
# try:
#     unido=inscricions.merge(quendas, on="clave_email", validate="m:1")
# except pd.errors.MergeError as erro:
#     print("Só co grupo:", erro)

unido = quendas.merge(inscricions, on=["clave_email"], how="left", validate="m:1", indicator=True)
unido["_merge"].value_counts()

_merge
both          700
left_only       4
right_only      0
Name: count, dtype: int64

Lista os correos que non casaron.

In [15]:
sin_unido = unido[unido["_merge"] == "left_only"]
sin_unido[["nome", "quenda", "clave_email","data","data_alta","data_nacemento","tarefa","horas"]]
#por puntos sabelalema@correo.example 
#por coreo carme.calvo@coreo.example
# porque no tendrá en la otra existente hasta ahora seguramente al ser nuevo - voluntario.novo@correo.example - no tiene inscripciones


,nome,quenda,clave_email,data,data_alta,data_nacemento,tarefa,horas
700,NaN,mañá,carme.calvo@coreo.example,2026-10-15,NaT,NaT,Reparto,3.0
701,NaN,mañá,carme.calvo@coreo.example,2026-10-15,NaT,NaT,Reparto,3.0
702,NaN,mañá,sabelalema@correo.example,2026-10-15,NaT,NaT,Reparto,3.0
703,NaN,mañá,voluntario.novo@correo.example,2026-10-15,NaT,NaT,Reparto,3.0


Para cada un, decide que fas e explícao: corrixilo (só se hai un único candidato claro) ou excluílo

In [16]:
quendas[quendas["clave_email"] == "carme.calvo@coreo.example"]

,data,quenda,tarefa,email,horas,clave_email
700,2026-10-15,mañá,Reparto,carme.calvo@coreo.example,3.0,carme.calvo@coreo.example
701,2026-10-15,mañá,Reparto,carme.calvo@coreo.example,3.0,carme.calvo@coreo.example


In [17]:
inscricions[inscricions["clave_email"] == "carme.calvo@coreo.example"]

,nome,email,data_nacemento,concello,data_alta,clave_email


In [18]:
inscricions[inscricions["clave_email"] == "sabela.lema@correo.example"]

,nome,email,data_nacemento,concello,data_alta,clave_email
58,Sabela Lema,sabela.lema@correo.example,1978-08-01,A Coruña,2024-05-17,sabela.lema@correo.example


In [19]:
quendas[quendas["clave_email"] == "sabela.lema@correo.example"]

,data,quenda,tarefa,email,horas,clave_email
127,2026-10-29,mañá,Clasificación,SABELA.LEMA@CORREO.EXAMPLE,2.5,sabela.lema@correo.example
463,2026-09-14,mañá,Reparto,sabela.lema@correo.example,3.0,sabela.lema@correo.example
566,2026-10-16,tarde,Recollida en supermercados,sabela.lema@correo.example,2.0,sabela.lema@correo.example
629,2026-10-18,tarde,Clasificación,SABELA.LEMA@CORREO.EXAMPLE,3.5,sabela.lema@correo.example


In [20]:
quendas[quendas["clave_email"] == "voluntario.novo@correo.example"]

,data,quenda,tarefa,email,horas,clave_email
703,2026-10-15,mañá,Reparto,voluntario.novo@correo.example,3.0,voluntario.novo@correo.example


In [21]:
inscricions[inscricions["clave_email"] == "voluntario.novo@correo.example"]

,nome,email,data_nacemento,concello,data_alta,clave_email


In [25]:
#corregir - remplazar en quendas el correo mal escrito por el correcto
# quendas.loc[quendas["clave_email"] == "carme.calvo@coreo.example"] = "carme.calvo@corero.example"
# quendas.loc[quendas["clave_email"] == "sabelalema@correo.example"] = "sabela.lema@correo.example"
quendas.loc[quendas["clave_email"] == "carme.calvo@coreo.example","clave_email"] = "carme.calvo@correo.example"
quendas.loc[quendas["clave_email"] == "sabelalema@correo.example","clave_email"] = "sabela.lema@correo.example"


In [26]:
unido_final = quendas.merge(inscricions, on=["clave_email"], how="left", validate="m:1", indicator=True)
unido_final["_merge"].value_counts()

_merge
both          703
left_only       1
right_only      0
Name: count, dtype: int64

In [29]:
sin_unido_final = unido_final[unido_final["_merge"] == "left_only"]
sin_unido_final[["nome", "quenda", "clave_email","data","data_alta","data_nacemento","tarefa","horas"]]

,nome,quenda,clave_email,data,data_alta,data_nacemento,tarefa,horas
703,NaN,mañá,voluntario.novo@correo.example,2026-10-15,NaT,NaT,Reparto,3.0


In [33]:
print(f"Excluídas: {len(sin_unido_final)} , {sin_unido_final['horas'].sum():.2f} horas")
#excluimos las que no tiene incripción 
# y las que tienen equivalencia pero está mal puesto el correo las volvemos a intentar mergerae formatenadolo otra vez 
# conserva únicamente las filas que han encontrado coincidencia entre quendas e inscricions
# elimina la columna auxiliar _merge, porque ya no la necesitas.
#unido = unido[unido["_merge"] == "both"].drop(columns="_merge")
unido_si = unido_final[unido_final["_merge"] == "both"].drop(columns="_merge")

#1 mergeamos otra vez
#2 luego eliminamos


Excluídas: 1 , 3.00 horas


Comproba que as horas incluídas máis as excluídas suman o total inicial.

In [36]:
total_horas= unido_si["horas"].sum() + sin_unido_final["horas"].sum() 
print(total_horas == horastotales)

True
